In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
import scanpy as sc
import scvi 
import plotnine as gg
import numpy as np
import matplotlib.pyplot as plt

from essential.utils import PLOTNINE_DEFAULT_THEME_2

In [ ]:
adata_path = "/workspace/data/260309_lce75_genomescale_ezrdm_glu_preprocessed.h5ad"
adata = sc.read_h5ad(adata_path)

In [ ]:
sc.pp.highly_variable_genes(adata, batch_key="batch", inplace=True, layer="reads", flavor="seurat_v3", n_top_genes=2000)
adata_ = adata[:, adata.var["highly_variable"]].copy()
adata_ = adata_[~adata_.obs["target"].isna()].copy()
adata_

In [ ]:
adata_

In [ ]:
adata_.obs["guide_purity"]

In [ ]:
# total number of UMIs across guides
adata_.obs["guide_n_umis"]

In [ ]:
# fraction of UMIs that map to the called guide
adata_.obs["guide_purity"]

In [ ]:
# target is not NA based on 
# of UMIs: something like >= 3
# and based on purity

In [ ]:
adata_.obs["top_target_unthresholded"]

In [ ]:
scvi.model.SCVI.setup_anndata(adata_, batch_key="batch", layer="reads")
model = scvi.model.SCVI(adata_)
model.train()

In [ ]:
latent = model.get_latent_representation()
adata_.obsm["latent"] = latent
sc.pp.neighbors(adata_, use_rep="latent")
sc.tl.umap(adata_)

adata_.obs["UMAP1"] = adata_.obsm["X_umap"][:, 0]
adata_.obs["UMAP2"] = adata_.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2", color="rt_bc"))
    + gg.geom_point(size=0.5, stroke=0.0)
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none"
    )
    + gg.labs(
        title="scVI latent space (no batch correction)"
    )
    
)


In [ ]:
model_bc = scvi.model.SCVI.setup_anndata(adata_, batch_key="rt_bc", layer="reads")
model_bc = scvi.model.SCVI(adata_)
model_bc.train()

In [ ]:
latent = model_bc.get_latent_representation()
adata_.obsm["latent"] = latent
sc.pp.neighbors(adata_, use_rep="latent")
sc.tl.umap(adata_)

adata_.obs["UMAP1"] = adata_.obsm["X_umap"][:, 0]
adata_.obs["UMAP2"] = adata_.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2", color="rt_bc"))
    + gg.geom_point(size=0.5, stroke=0.0)
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none"
    )
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    
)


In [ ]:
(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point(size=0.5, stroke=0.0)
    + gg.geom_point(adata_.obs.query("target == 'nontargeting'"), size=2, stroke=0.0, color="red")
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none"
    )
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    
)

In [ ]:
adata_.layers["reads"]

In [ ]:
adata_.X = adata_.layers["reads"]
sc.pp.normalize(adata_)
sc.

In [ ]:
import plotly.express as px
import pandas as pd


fig = px.scatter(adata_.obs, x="UMAP1", y="UMAP2", color="target", hover_data=["target"],)
fig.update_traces(marker=dict(size=3))
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
sc.pp.neighbors(adata_, use_rep="latent")
sc.tl.umap(adata_)


In [ ]:
adata_.obs["UMAP1"] = adata_.obsm["X_umap"][:, 0]
adata_.obs["UMAP2"] = adata_.obsm["X_umap"][:, 1]


In [ ]:
sc.tl.leiden(adata_, resolution=1)

In [ ]:
sc.pl.umap(adata_, color=["leiden"])

In [ ]:
(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point(size=0.5, stroke=0.0, color="grey")
    + gg.geom_point(adata_.obs.query("target == 'nontargeting'"), size=2, stroke=0.0, color="red")
)

In [ ]:
n_leiden_clusters = adata_.obs.groupby("target")["leiden"].nunique().to_frame("n_leiden_clusters")

(
    gg.ggplot(n_leiden_clusters, gg.aes(x="n_leiden_clusters"))
    + gg.geom_histogram(binwidth=1)
)



In [ ]:
fracs = adata_.obs.groupby("target")["leiden"].value_counts(normalize=True)
top_fracs = fracs.groupby("target").max().to_frame("top_frac")

(
    gg.ggplot(top_fracs, gg.aes(x="top_frac"))
    + gg.geom_histogram(binwidth=0.1)
)


In [ ]:
for leiden_cluster in range(len(adata_.obs["leiden"].unique())):
    leiden_cluster_str = str(leiden_cluster)
    counts_ = adata_[adata_.obs["leiden"] == leiden_cluster_str].obs["target"].value_counts()
    print("Cluster", leiden_cluster_str)
    print(counts_.head(25))
    print()

In [ ]:
renamer = {
  "0": "mixed metabolism (weak)",
  "1": "uncharacterized/baseline (weak)",
  "2": "envelope/surface (weak)",
  "3": "acid resistance and biosynthesis (weak)",
  "4": "mixed metabolism (weak)",
  "5": "stress response (weak)",
  "6": "mixed (weak)",
  "7": "envelope and transport (weak)",
  "8": "envelope/biofilm (weak)",
  "9": "mixed (weak)",
  "10": "mixed (weak)",
  "11": "mixed biosynthesis (weak)",
  "12": "respiration and prosthetic-group biosynthesis",
  "13": "transcription and translation",
  "14": "cell envelope biogenesis",
  "15": "uncharacterized (weak)",
  "16": "cell division",
  "17": "iron acquisition (enterobactin/Fur)",
  "18": "phosphate uptake and chaperonin",
  "19": "Rho-dependent transcription termination"
}
adata_.obs["cluster_name"] = adata_.obs["leiden"].map(renamer).values

In [ ]:
(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2", color="cluster_name")) 
    + gg.geom_point(size=0.5, stroke=0.0) 
    + gg.theme_classic()
    + gg.theme(

    )
)


In [ ]:
import pymde

In [ ]:
adata_select = adata_[~adata_.obs["target"].isna()]
latent = adata_select.obsm["latent"]
latent

In [ ]:
mde = pymde.preserve_neighbors(
    latent,
    embedding_dim=2,
    n_neighbors=15,          # ↑ more = more global structure preserved
    repulsive_fraction=0.5,  # fraction of non-neighbor pairs to use
    init="quadratic",        # good default; "random" for experimentation
    device="cuda",           # if GPU available
)
embedding = mde.embed(verbose=True)

In [ ]:
adata_select.obsm["mde"] = embedding.cpu().numpy()
plot_df["mde_1"] = adata_select.obsm["mde"][:, 0]
plot_df["mde_2"] = adata_select.obsm["mde"][:, 1]



In [ ]:
(
    gg.ggplot(adata.obs, gg.aes(x="mde_1", y="mde_2", color="target")) 
    + gg.geom_point(size=0.5, stroke=0.0) 
    + gg.theme_classic()
    + gg.theme(
        legend_position="none"
    )
)


In [ ]:
plot_df = adata_select.obs.copy()

mde_vals = embedding.cpu().numpy().copy()
mde0 = mde_vals[:, 0]
mde1 = mde_vals[:, 1]
# qmin = 0.01
# qmax = 0.99
# mde0_clip = np.clip(mde0, np.quantile(mde0, qmin), np.quantile(mde0, qmax))
# mde1_clip = np.clip(mde1, np.quantile(mde1, qmin), np.quantile(mde1, qmax))

plot_df["mde_1"] = mde0
plot_df["mde_2"] = mde1


(
    gg.ggplot(plot_df, gg.aes(x="mde_1", y="mde_2", color="target")) 
    + gg.geom_point(size=0.5, stroke=0.0) 
    + gg.theme_classic()
    + gg.theme(
        legend_position="none"
    )
)


In [ ]:
import plotly.express as px
import pandas as pd


fig = px.scatter(plot_df, x="mde_1", y="mde_2", color="target", hover_data=["target"],)
fig.update_traces(marker=dict(size=3))
fig.update_layout(showlegend=False)
fig.show()